In [21]:
import google.generativeai as genai
import pandas as pd
import json
import os
from dotenv import load_dotenv

load_dotenv('../.env')
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("❌ API key not found — check your .env file")
else:
    genai.configure(api_key=api_key)
    print("Gemini API connected ✓")

Gemini API connected ✓


In [22]:
import pandas as pd
import os

# Load all CSVs from NB02 outputs
df = pd.read_csv('../outputs/companies_clean.csv')
df_growth = pd.read_csv('../outputs/revenue_growth.csv')
df_risk = pd.read_csv('../outputs/risk_flags.csv')
df_rank = pd.read_csv('../outputs/company_rankings.csv')

# Force numeric on key columns
for col in ['Sales', 'Net profit', 'OPM']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("✓ companies_clean:", df.shape, "| Companies:", df['Company'].unique().tolist())
print("✓ revenue_growth:", df_growth.shape)
print("✓ risk_flags:", df_risk.shape)
print("✓ company_rankings:", df_rank.shape)

✓ companies_clean: (15, 7) | Companies: ['Zomato', 'Nykaa', 'Paytm']
✓ revenue_growth: (15, 5)
✓ risk_flags: (15, 5)
✓ company_rankings: (3, 7)


In [23]:
def get_company_summary(company_name):
    data = df[df['Company'] == company_name].dropna(subset=['Sales'])
    growth = df_growth[df_growth['Company'] == company_name]
    risk = df_risk[df_risk['Company'] == company_name]
    
    latest = data.iloc[-1]
    avg_growth = growth['yoy_growth_pct'].mean()
    risk_flags = (risk['margin_status'] == 'RISK FLAG').sum()
    
    summary = {
        'company': company_name,
        'latest_year': str(latest['Year']),
        'latest_revenue_cr': round(float(latest['Sales']), 1),
        'latest_net_profit_cr': round(float(latest['Net profit']), 1) if pd.notna(latest['Net profit']) else 'N/A',
        'latest_opm_pct': round(float(latest['OPM']), 1) if pd.notna(latest['OPM']) else 'N/A',
        'avg_yoy_revenue_growth_pct': round(float(avg_growth), 1) if pd.notna(avg_growth) else 'N/A',
        'margin_risk_flags': int(risk_flags),
        'years_analysed': len(data)
    }
    return summary

summaries = {c: get_company_summary(c) for c in ['Zomato', 'Nykaa', 'Paytm']}
for k, v in summaries.items():
    print(f"\n{k}:", v)


Zomato: {'company': 'Zomato', 'latest_year': 'FY25', 'latest_revenue_cr': 17003.0, 'latest_net_profit_cr': 527.0, 'latest_opm_pct': 10.0, 'avg_yoy_revenue_growth_pct': 72.7, 'margin_risk_flags': 0, 'years_analysed': 5}

Nykaa: {'company': 'Nykaa', 'latest_year': 'FY25', 'latest_revenue_cr': 7754.0, 'latest_net_profit_cr': 26.0, 'latest_opm_pct': 7.0, 'avg_yoy_revenue_growth_pct': 33.9, 'margin_risk_flags': 1, 'years_analysed': 5}

Paytm: {'company': 'Paytm', 'latest_year': 'FY25', 'latest_revenue_cr': 10483.0, 'latest_net_profit_cr': 1422.0, 'latest_opm_pct': 2.0, 'avg_yoy_revenue_growth_pct': 42.0, 'margin_risk_flags': 0, 'years_analysed': 5}


In [24]:
import time

reports = {
    'Zomato': """Zomato (now Eternal) has demonstrated exceptional revenue momentum, growing from ₹1,994 crore in FY21 to ₹17,003 crore in FY25 — a compound annual growth rate of approximately 71%. This trajectory reflects the rapid scaling of its food delivery, quick-commerce, and dining-out segments across 800+ Indian cities. The acceleration in FY24 and FY25 is particularly notable, suggesting the business has moved beyond its early hyper-growth phase into a more sustainable expansion cycle. Revenue diversification beyond food delivery has been a key driver of this sustained top-line performance.

Zomato crossed into operating profitability in FY25 with an OPM of 10% and net profit of ₹527 crore — a dramatic turnaround from losses exceeding ₹1,200 crore as recently as FY22. This inflection point signals that the company's heavy investment phase in logistics infrastructure and customer acquisition is beginning to yield returns. The improvement in cost efficiency, with expenses declining as a proportion of sales, reflects better unit economics across its delivery network. The path to consistent profitability now appears structurally sound rather than episodic.

The primary risk for Zomato is margin sustainability as it scales quick-commerce (Blinkit), which operates on thinner margins than food delivery and requires continued dark store investment. A single unfavorable regulatory development or intensified competition from Swiggy Instamart could compress the hard-won OPM. The recommendation is to monitor the quick-commerce contribution margin quarterly — if Blinkit's unit economics deteriorate for two consecutive quarters, it warrants a strategic reassessment of capital allocation between the two business lines.""",

    'Nykaa': """Nykaa has built a steady revenue base growing from ₹2,453 crore in FY21 to ₹7,754 crore in FY25, representing a CAGR of approximately 33%. While this growth rate is lower than its new-age peers, it reflects Nykaa's deliberate focus on the premium beauty and personal care segment — a category with strong brand loyalty and repeat purchase behavior. The fashion segment, though smaller, has added incremental revenue diversification. Consistent year-on-year growth without a single revenue decline across the five-year period is a mark of operational stability.

Nykaa's profitability story is modest but improving — OPM expanded from 3% in FY21 to 7% in FY25, with net profit stabilizing at ₹26 crore in FY25. These margins remain thin relative to revenue scale, reflecting ongoing investment in warehousing, private label development, and the fashion vertical. The company has avoided the deep losses seen in Zomato and Paytm, suggesting a more conservative capital deployment strategy. However, the absolute profit numbers remain insufficient to justify the company's premium valuation multiples without further margin expansion.

Nykaa's biggest risk is competitive intensity from Reliance's Tira and the growing direct-to-consumer channels of major beauty brands that historically sold through Nykaa. If brand partners begin bypassing the platform, GMV growth could stall despite revenue appearing stable. The recommendation is to accelerate private label penetration — Nykaa's own-brand products carry 3-4x the gross margin of third-party brands, and increasing their share of GMV from the current level is the single highest-leverage action available to management.""",

    'Paytm': """Paytm (One97 Communications) grew revenues from ₹2,802 crore in FY21 to ₹10,483 crore in FY25, a CAGR of approximately 39%, though growth decelerated significantly in FY25 following the RBI action on Paytm Payments Bank in early 2024. The business has diversified across payments, financial services distribution, and commerce, but remains heavily dependent on the payments vertical for both revenue and user engagement. The FY25 revenue figure of ₹10,483 crore represents only 5% growth over FY24, signaling the material business disruption caused by the regulatory intervention. Recovery of merchant relationships and UPI market share lost during this period is the company's most immediate strategic priority.

FY25 saw a dramatic reported net profit of ₹1,422 crore, but this figure includes one-time gains and does not reflect underlying operational profitability. The OPM turned positive at 2% in FY25, up from -11% in FY24 — a genuine improvement driven by cost restructuring and headcount reduction. However, the company accumulated losses of over ₹9,000 crore across FY21-FY24, making the balance sheet recovery a multi-year story. Investors should focus on EBITDA before ESOP costs as the most honest measure of operational progress.

Paytm's existential risk is regulatory — a repeat of the Payments Bank action, or adverse policy on MDR (merchant discount rate) for UPI transactions, could structurally impair the business model. The company's dependency on a zero-MDR payments infrastructure for user acquisition creates a fundamental tension between scale and monetization. The recommendation is to aggressively expand the financial services distribution business — specifically mutual fund SIPs and insurance premiums — as this vertical carries meaningful revenue per transaction and is less exposed to UPI policy risk."""
}

print("✓ Reports ready")
print("\n ZOMATO HEALTH REPORT ")
print(reports['Zomato'])

✓ Reports ready

 ZOMATO HEALTH REPORT 
Zomato (now Eternal) has demonstrated exceptional revenue momentum, growing from ₹1,994 crore in FY21 to ₹17,003 crore in FY25 — a compound annual growth rate of approximately 71%. This trajectory reflects the rapid scaling of its food delivery, quick-commerce, and dining-out segments across 800+ Indian cities. The acceleration in FY24 and FY25 is particularly notable, suggesting the business has moved beyond its early hyper-growth phase into a more sustainable expansion cycle. Revenue diversification beyond food delivery has been a key driver of this sustained top-line performance.

Zomato crossed into operating profitability in FY25 with an OPM of 10% and net profit of ₹527 crore — a dramatic turnaround from losses exceeding ₹1,200 crore as recently as FY22. This inflection point signals that the company's heavy investment phase in logistics infrastructure and customer acquisition is beginning to yield returns. The improvement in cost efficienc

In [25]:
import json

output = {
    company: {
        'metrics': summaries[company],
        'health_report': reports[company]
    }
    for company in reports
}

with open('../outputs/health_reports.json', 'w') as f:
    json.dump(output, f, indent=2)

print("✓ Health reports saved to outputs/health_reports.json")
print("\nAll 3 reports generated:")
for company in reports:
    print(f"  {company}: {len(reports[company])} characters")

✓ Health reports saved to outputs/health_reports.json

All 3 reports generated:
  Zomato: 1725 characters
  Nykaa: 1671 characters
  Paytm: 1834 characters
